# Chapter 7
## Linear Integrate-and-Fire (LIF) Neurons
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter07.ipynb)

## About this chapter

This chapter compares the full Hodgkin-Huxley (HH) neuron's subthreshold
response with the linear integrate-and-fire (LIF) neuron -- a reduced model
that discards explicit ionic gates and instead resets a linear voltage
variable at a threshold.

Below threshold, the HH voltage equation is approximately linear, so an
effective membrane time constant and a passive current decomposition are
meaningful even though the full model is nonlinear. The LIF model keeps only
that linear decay and adds a hard reset: it forgets everything about gating
kinetics, refractoriness, and spike shape, trading detail for a closed-form,
easily analyzed spike train.

The HH subthreshold response follows the usual conductance-based balance

$$
C\frac{dV}{dt}=I_{\mathrm{ext}}-g_{\mathrm{Na}}m^3h(V-E_{\mathrm{Na}})
-g_{\mathrm{K}}n^4(V-E_{\mathrm{K}})-g_{\mathrm{L}}(V-E_{\mathrm{L}}).
$$

The LIF neuron instead follows

$$
\tau_m\frac{dV}{dt}=-V+\tau_m I,\qquad V\ge1\Rightarrow V\leftarrow0.
$$

Here $V$ is a normalized voltage, $\tau_m$ is the membrane time constant,
and $I$ is a constant drive; once $V$ reaches the threshold $1$ it resets to
$0$.

See [`README.md`](chapter07.md) for
the full guide, including suggested order and related chapters.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact

## The Hodgkin-Huxley Neuron's Subthreshold Response

`m`, `h`, and `n` all obey the classical HH kinetics; this one simulation
backs the LIF-vs-HH voltage comparison, the current decomposition, and the
effective time constant below.

In [ ]:
def simulate_hh_subthreshold(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                              v_k=-82.0, v_na=45.0, v_l=-59.0,
                              i_ext=7.0, t_final=100.0, dt=0.01):
    def alpha_h(v):
        return 0.07 * exp(-(v + 70) / 20)

    def alpha_m(v):
        return (v + 45) / 10.0 / (1 - exp(-(v + 45) / 10))

    def alpha_n(v):
        return 0.01 * (-60.0 - v) / (exp((-60 - v) / 10) - 1)

    def beta_h(v):
        return 1. / (exp(-(v + 40) / 10) + 1)

    def beta_m(v):
        return 4 * exp(-(v + 70) / 18)

    def beta_n(v):
        return 0.125 * exp(-(v + 70) / 80)

    def h_inf(v):
        return alpha_h(v) / (alpha_h(v) + beta_h(v))

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def n_inf(v):
        return alpha_n(v) / (alpha_n(v) + beta_n(v))

    def derivative(x0, t):
        v, m, n, h = x0
        i_na = -g_na * h * m ** 3 * (v - v_na)
        i_k = -g_k * n ** 4 * (v - v_k)
        i_l = -g_l * (v - v_l)
        dv = (i_ext + i_na + i_k + i_l) / c
        dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return [dv, dm, dn, dh]

    v0 = -20.0
    x0 = [v0, m_inf(v0), n_inf(v0), h_inf(v0)]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0], sol[:, 1], sol[:, 2], sol[:, 3]

### Figure 7.1 -- HH voltage vs. the LIF linear approximation

In [ ]:
def plot_lif_neuron_with_hh(t, v):
    plt.figure(figsize=(7, 3))

    v_lin = np.asarray([-82.0, -54.0])
    t_lin = np.asarray([0.0, (v_lin[1] - v_lin[0]) / 1.65])
    plt.plot(t_lin, v_lin, c="b", lw=2)
    for i in range(1, 6):
        plt.plot(t_lin + t_lin[1] * i, v_lin, c="b", lw=2)
        plt.plot([i * t_lin[1], i * t_lin[1]], v_lin, c="b", lw=2, ls="--")

    plt.plot(t, v, lw=2, c="k")
    plt.xlim(min(t), max(t))
    plt.ylim(-100, 50)
    plt.xlabel("time [ms]")
    plt.ylabel("v [mV]")
    plt.yticks(range(-100, 100, 50))
    plt.tight_layout()
    plt.show()


t, v, m, n, h = simulate_hh_subthreshold()
plot_lif_neuron_with_hh(t, v)

### Figure 7.2 -- effective membrane time constant

In [ ]:
def plot_tau_m_for_hh(t, m, h, n, g_na=120.0, g_k=36.0, g_l=0.3):
    tau = 1 / (g_k * n ** 4 + g_na * m ** 3 * h + g_l)
    plt.figure(figsize=(7, 3))
    plt.plot(t, tau, lw=2, c="k")
    plt.xlim(min(t), max(t))
    plt.xlabel("time [ms]")
    plt.ylabel(r"$\tau$ [ms]")
    plt.tight_layout()
    plt.show()


plot_tau_m_for_hh(t, m, h, n)

### Figure 7.3 -- ionic current decomposition

In [ ]:
def plot_subthr_for_hh(t, v, m, h, n, g_na=120.0, g_k=36.0, g_l=0.3,
                       v_na=45.0, v_k=-82.0, v_l=-59.0):
    i_na = g_na * m ** 3 * h * (v_na - v)
    i_k = g_k * n ** 4 * (v_k - v)
    i_l = g_l * (v_l - v)
    i_tot = i_k + i_na + i_l

    plt.figure(figsize=(7, 3))
    plt.plot(t, i_na, lw=2, c="r", label=r"$I_{na}$")
    plt.plot(t, i_k, lw=2, c="g", label=r"$I_{k}$")
    plt.plot(t, i_l, lw=2, c="b", label=r"$I_{l}$")
    plt.plot(t, i_tot, lw=2, c='k', label=r"$I_{tot}$")
    plt.legend()
    plt.xlim(min(t), max(t))
    plt.ylim(-20, 20)
    plt.xlabel("time [ms]", fontsize=15)
    plt.ylabel(r"$I\ [\mu A/cm^2]$", fontsize=15)
    plt.tight_layout()
    plt.tick_params(labelsize=15)
    plt.show()


plot_subthr_for_hh(t, v, m, h, n)

In [ ]:
interact(lambda i_ext=7.0: plot_lif_neuron_with_hh(*simulate_hh_subthreshold(i_ext=i_ext)[:2]),
         i_ext=(0.0, 15.0, 0.5));

## The Linear Integrate-and-Fire (LIF) Neuron

A pure RK4-integrated linear decay with a hard reset at `v=1`. `tau_m` and
`i` select between the two book figures.

In [ ]:
def integrate_rk4(x, dt, f):
    k1 = dt * f(x)
    k2 = dt * f(x + 0.5 * k1)
    k3 = dt * f(x + 0.5 * k2)
    k4 = dt * f(x + k3)
    return x + (k1 + 2.0 * (k2 + k3) + k4) / 6.0


def simulate_lif_neuron(tau_m=10.0, i=0.11, t_final=100.0, dt=0.01):
    def derivative(v):
        return -v / tau_m + i

    num_steps = int(t_final / dt)
    v = np.zeros(num_steps)
    t = np.arange(0, t_final, dt)
    for k in range(1, num_steps):
        v_new = integrate_rk4(v[k - 1], dt, derivative)
        v[k] = v_new if v_new <= 1 else 0.0
    return t, v


def plot_lif_neuron(t, v):
    plt.figure(figsize=(7, 3))
    plt.xlabel("time [ms]", fontsize=14)
    plt.ylabel("v [mV]", fontsize=14)
    plt.ylim([0, 2])
    plt.xlim(0, max(t))
    plt.tight_layout()
    plt.plot(t, v, lw=2, c="k")
    plt.show()

### Figure 7.4

In [ ]:
plot_lif_neuron(*simulate_lif_neuron(tau_m=10.0, i=0.11))

### Figure 7.5

`i` is chosen so the analytic period is exactly 20 ms at `tau_m=2`.

In [ ]:
tau_m2 = 2.0
i2 = 1 / (1 - np.exp(-20.0 / tau_m2)) / tau_m2
plot_lif_neuron(*simulate_lif_neuron(tau_m=tau_m2, i=i2, t_final=50.0))

In [ ]:
interact(lambda tau_m=10.0, i=0.11: plot_lif_neuron(*simulate_lif_neuron(tau_m=tau_m, i=i)),
         tau_m=(0.5, 20.0, 0.5), i=(0.0, 1.0, 0.01));